# 🛡️ MayaGuard - QLoRA Fine-Tuning: Hallucination-Aware LLM

This notebook fine-tunes **Mistral-7B-Instruct-v0.2** with **QLoRA** (4-bit quantization
+ LoRA adapters) to produce more grounded, citation-aware responses for the
MayaGuard safety pipeline.

**Runtime:** Google Colab Free Tier (T4 GPU, 16GB VRAM)
**Training Time:** ~45-90 minutes
**Base Model:** `mistralai/Mistral-7B-Instruct-v0.2` (7B params, loaded in 4-bit)
**QLoRA Config:** Rank 64, Alpha 16, 4-bit NF4 quantization

---

### Memory Budget (T4 16GB)
| Component | VRAM |
|-----------|------|
| Base model (4-bit NF4) | ~4.5 GB |
| LoRA adapters | ~0.3 GB |
| Optimizer states | ~1.5 GB |
| Activations + gradients | ~4-6 GB |
| **Total** | **~10-12 GB** |

---

## 1. Install Dependencies

In [ ]:
!pip install -q transformers peft bitsandbytes trl datasets accelerate torch

In [ ]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    raise RuntimeError("GPU not available! Switch to a GPU runtime in Colab.")

## 2. Upload Training Data

Run `python -m finetuning.data.generate_instruct_dataset` locally first,
then upload `instruct_train.jsonl` and `instruct_val.jsonl`.

In [ ]:
from google.colab import files
import os

os.makedirs("data", exist_ok=True)

print("Upload instruct_train.jsonl and instruct_val.jsonl:")
uploaded = files.upload()

for name, content in uploaded.items():
    with open(f"data/{name}", "wb") as f:
        f.write(content)
    print(f"  Saved: data/{name} ({len(content)} bytes)")

## 3. Load and Inspect Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={
        "train": "data/instruct_train.jsonl",
        "validation": "data/instruct_val.jsonl",
    },
)

print(f"Train samples: {len(dataset['train'])}")
print(f"Val samples:   {len(dataset['validation'])}")
print(f"\nDomain distribution (train):")

domains = {}
for ex in dataset["train"]:
    d = ex.get("domain", "unknown")
    domains[d] = domains.get(d, 0) + 1
for domain, count in sorted(domains.items()):
    print(f"  {domain}: {count}")

print(f"\nSample message structure:")
sample = dataset["train"][0]
for msg in sample["messages"]:
    print(f"  [{msg['role']}]: {msg['content'][:100]}...")

## 4. Load Base Model in 4-bit (QLoRA)

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_NAME = "mistralai/Mistral-7B-Instruct-v0.2"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",             # Normalized Float 4-bit
    bnb_4bit_compute_dtype=torch.float16,   # Compute in FP16
    bnb_4bit_use_double_quant=True,         # Nested quantization
)

print(f"Loading {MODEL_NAME} in 4-bit...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model.config.use_cache = False  # Required for gradient checkpointing

total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel loaded! Total parameters: {total_params:,}")
print(f"GPU memory used: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

## 5. Apply QLoRA Configuration

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model

# Prepare model for k-bit training
model = prepare_model_for_kbit_training(model)

# QLoRA configuration targeting attention projection layers
qlora_config = LoraConfig(
    r=64,                                    # LoRA rank
    lora_alpha=16,                           # Scaling factor
    lora_dropout=0.1,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",  # Attention projections
        "gate_proj", "up_proj", "down_proj",       # MLP layers
    ],
    bias="none",
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, qlora_config)

# Print trainable parameters
model.print_trainable_parameters()
print(f"GPU memory after LoRA: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

## 6. Format Dataset for SFTTrainer

In [ ]:
def format_chat(example):
    """Format messages into Mistral chat template."""
    messages = example["messages"]
    formatted = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False,
    )
    return {"text": formatted}

formatted_dataset = dataset.map(format_chat, remove_columns=dataset["train"].column_names)

print(f"Sample formatted text (first 500 chars):")
print(formatted_dataset["train"][0]["text"][:500])
print("...")

## 7. Training Configuration

In [ ]:
from trl import SFTTrainer, SFTConfig

OUTPUT_DIR = "mayaguard-qlora-mistral"

sft_config = SFTConfig(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=2,           # Small batch for T4 memory
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,           # Effective batch size = 8
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    max_seq_length=1024,
    eval_strategy="steps",
    eval_steps=50,
    save_strategy="steps",
    save_steps=100,                          # Checkpoint every 100 steps (Colab safety)
    save_total_limit=3,
    logging_steps=10,
    fp16=True,
    gradient_checkpointing=True,             # Essential for T4 memory
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",               # Memory-efficient optimizer
    report_to="none",
    dataset_text_field="text",
    packing=False,
)

print(f"Training configuration:")
print(f"  Epochs: {sft_config.num_train_epochs}")
print(f"  Batch size (per device): {sft_config.per_device_train_batch_size}")
print(f"  Gradient accumulation: {sft_config.gradient_accumulation_steps}")
print(f"  Effective batch size: {sft_config.per_device_train_batch_size * sft_config.gradient_accumulation_steps}")
print(f"  Max seq length: {sft_config.max_seq_length}")
print(f"  Gradient checkpointing: {sft_config.gradient_checkpointing}")
print(f"  Optimizer: {sft_config.optim}")

## 8. Train with SFTTrainer

In [ ]:
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=formatted_dataset["train"],
    eval_dataset=formatted_dataset["validation"],
    tokenizer=tokenizer,
)

print("Starting QLoRA fine-tuning...")
print(f"GPU memory before training: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")

# Resume from checkpoint if available (Colab disconnect safety)
import os
checkpoints = [d for d in os.listdir(OUTPUT_DIR) if d.startswith("checkpoint")] if os.path.exists(OUTPUT_DIR) else []
resume_from = None
if checkpoints:
    latest = sorted(checkpoints, key=lambda x: int(x.split("-")[-1]))[-1]
    resume_from = os.path.join(OUTPUT_DIR, latest)
    print(f"Resuming from checkpoint: {resume_from}")

train_result = trainer.train(resume_from_checkpoint=resume_from)

print(f"\nTraining complete!")
print(f"  Total steps: {train_result.global_step}")
print(f"  Training loss: {train_result.training_loss:.4f}")
print(f"  GPU memory peak: {torch.cuda.max_memory_allocated() / 1024**3:.2f} GB")

## 9. Save QLoRA Adapter

In [ ]:
ADAPTER_DIR = "mayaguard-qlora-mistral/final_adapter"

# Save only the LoRA adapter weights
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# Check adapter size
adapter_size = sum(
    os.path.getsize(os.path.join(ADAPTER_DIR, f))
    for f in os.listdir(ADAPTER_DIR)
    if os.path.isfile(os.path.join(ADAPTER_DIR, f))
)
print(f"\nQLoRA adapter saved to: {ADAPTER_DIR}")
print(f"Adapter size: {adapter_size / 1024 / 1024:.2f} MB")
print(f"(vs base model ~14 GB - {adapter_size / (14 * 1024 * 1024 * 1024) * 100:.2f}% of original)")

## 10. Test Generation with Fine-tuned Model

In [ ]:
# Re-enable cache for inference
model.config.use_cache = True

def generate_answer(query: str, context: str = "") -> str:
    """Generate an answer using the fine-tuned model."""
    messages = [
        {
            "role": "system",
            "content": (
                "You are a trustworthy AI assistant. Always ground your answers in the "
                "provided reference documents. Cite sources explicitly. If the documents "
                "do not contain enough evidence, refuse to answer."
            ),
        },
        {"role": "user", "content": f"{query}\n\nContext:\n{context}"},
    ]
    
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(formatted, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.3,
            do_sample=True,
            top_p=0.9,
        )
    
    # Decode only the generated part
    response = tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
    return response.strip()


# Test cases
context = (
    "[1] (Source: PubMed:PMC8122345) Metformin is the primary pharmacotherapy for type 2 "
    "diabetes mellitus. It operates by activating AMPK in hepatocytes.\n\n"
    "[2] (Source: FDA Warning:Metformin) Lactic acidosis is a rare but life-threatening "
    "adverse drug reaction associated with Metformin accumulation."
)

test_queries = [
    "What is the mechanism of action of Metformin?",
    "Can Metformin cure cancer?",
    "What are the side effects of Metformin?",
]

print("\n" + "=" * 80)
print("INFERENCE TEST RESULTS")
print("=" * 80)

for query in test_queries:
    print(f"\n📋 Query: {query}")
    print("-" * 60)
    answer = generate_answer(query, context)
    print(f"🤖 Answer: {answer}")
    print()

## 11. Download Adapter

Download the adapter weights to your local machine. Place them in
`mayaguard/finetuning/outputs/qlora_adapter/` and configure:

```
QLORA_ADAPTER_PATH=finetuning/outputs/qlora_adapter
QLORA_ADAPTER_ENABLED=true
```

**Note:** To serve this adapter with vLLM, start the server with:
```bash
python -m vllm.entrypoints.openai.api_server \\
    --model mistralai/Mistral-7B-Instruct-v0.2 \\
    --enable-lora \\
    --lora-modules mayaguard=finetuning/outputs/qlora_adapter
```

In [ ]:
import shutil

shutil.make_archive("qlora_adapter", "zip", ADAPTER_DIR)
files.download("qlora_adapter.zip")

print("\nDownload complete! Next steps:")
print("1. Unzip to mayaguard/finetuning/outputs/qlora_adapter/")
print("2. Set QLORA_ADAPTER_PATH=finetuning/outputs/qlora_adapter")
print("3. Set QLORA_ADAPTER_ENABLED=true")
print("4. Restart MayaGuard or serve with vLLM --enable-lora")

## 12. (Optional) Push to HuggingFace Hub

In [ ]:
# Uncomment and fill in your HuggingFace credentials
# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN_HERE")
#
# model.push_to_hub("your-username/mayaguard-qlora-mistral")
# tokenizer.push_to_hub("your-username/mayaguard-qlora-mistral")
#
# print("QLoRA adapter pushed to HuggingFace Hub!")